# Clase 3 · 06 · Visualización orientada a preguntas
Ejecutar manualmente después de stream_005 y su prueba de idempotencia.
Para cada consigna incluí pregunta, agregación, gráfico y una conclusión de dos o tres oraciones.
Spark agrega los datos; sólo el resultado pequeño se convierte a Pandas para dibujar.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
require_stream_lab(spark, config)

In [ ]:
assert spark.table("stream_run_audit").where("expected_batch_id = 'stream_005'").count() > 0, "Terminá las cinco etapas primero."
import matplotlib.pyplot as plt

## Ejemplo resuelto — Ventas por canal
¿Qué canal concentra el monto de las ventanas finalizadas?

In [ ]:
sales = (spark.table("gold_stream_windows").groupBy("payment_channel")
         .agg(F.sum("total_amount").alias("total_amount")).orderBy("payment_channel"))
plot_data = sales.toPandas()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(plot_data["payment_channel"], plot_data["total_amount"].astype(float), color="#2563eb")
ax.set_title("Monto de compras por canal · ventanas finalizadas")
ax.set_xlabel("Canal")
ax.set_ylabel("Monto total ficticio")
fig.tight_layout()
plt.show()
display(sales)
leader = sales.orderBy(F.desc("total_amount")).first()
print(f"Respuesta: {leader.payment_channel} concentra el mayor monto ({leader.total_amount}).")

## Consigna 1 — Evolución por ventanas
¿Qué ventana tiene el mayor monto y cuál tiene la mayor cantidad de compras? ¿Coinciden?
Agregá por window_start y separá las unidades de ambos ejes o usá dos paneles.
Fuente: gold_stream_windows. Aclarar que muestra ventanas finalizadas, no todas las llegadas.

In [ ]:
# TODO: agregación, gráfico con unidades y respuesta escrita.

## Consigna 2 — Fraude y tamaño de muestra
¿Qué canal tiene mayor tasa global de fraude sintético? ¿Cuántas compras respaldan cada tasa?
Usá sum(fraud_count) / sum(event_count), no el promedio de tasas por ventana.
Mostrá tasa y volumen. Fuente: gold_stream_windows.

In [ ]:
# TODO: agregación, visualización y respuesta escrita.

## Consigna 3 — Llegadas y tiempo del evento
¿En qué lote llegan late_ok y late_bad y qué antigüedad tienen respecto de los eventos normales?
Mostrá la distribución de event_ts por source_batch_id usando silver_stream_valid.
La posición del lote es el orden de publicación; ingested_at es el momento real de procesamiento.

In [ ]:
# TODO: distribución temporal o gráfico de puntos con etiquetas y explicación.

## Consigna 4 — Watermark y cierre de ventanas
¿Cómo cambia Gold entre las etapas y qué evidencia muestra el avance del watermark?
Usá las auditorías para los conteos de Gold y stream_query_progress para watermarks y descartes.
Evitá contar reejecuciones como nuevas llegadas. Un gráfico por etapa y una tabla de evidencia son suficientes.

In [ ]:
# TODO: seleccionar una auditoría representativa por etapa, gráfico y evidencia del progreso.